# swiftmap quickstart

The whole journey in one notebook: a DataFrame onto a map, colored by a value,
sized by another, organized into sidebar folders, and exported as one file you can
hand to anyone.

Needs only `swiftmap` and `pandas`. Every dataset is generated in-cell — nothing to
download.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 250

# Sensor readings scattered around the Strait of Gibraltar
df = pd.DataFrame({
    "lat": 36.02 + rng.normal(0, 0.06, n),
    "lon": -5.45 + rng.normal(0, 0.10, n),
    "site": [f"Sensor {i:03d}" for i in range(n)],
    "reading": np.round(rng.gamma(4, 4, n), 1),
    "volume": rng.integers(10, 500, n),
    "status": rng.choice(["Active", "Idle", "Fault"], n, p=[0.6, 0.3, 0.1]),
})
df.head()

## One call, one map

`add_circle_markers` finds the `lat`/`lon` columns by name (`latitude`/`longitude`,
`x`/`y`, and `lng` work too; `lat_col=`/`lon_col=` override the guess).
`color_col="reading"` colors each point through a colormap — viridis by default.

And because no `center` or `zoom` was given, **the map fits itself to the data**.

In [ ]:
from swiftmap import Map

m = Map()
m.add_circle_markers(df, name="Sensors", color_col="reading")
m

Hover a point for a tooltip, click for a popup — every column shows by default;
narrowing that down comes later in this notebook. The sidebar at the top right
toggles layers and basemaps.

## Size by a value too

`radius_col` sizes each point so **area** is proportional to the value — a doubled
value looks doubled, not quadrupled. `radius_range` bounds the pixel radii.

In [ ]:
m2 = Map()
m2.add_circle_markers(df, name="Sensors", color_col="reading",
                      radius_col="volume", radius_range=(3, 16))
m2

## Fan out into sidebar folders

Any part of `layer_group` that matches a column name resolves per row, so one call
builds a folder tree from the data. A non-numeric `color_col` is categorical: each
distinct value takes a palette color automatically.

In [ ]:
m3 = Map()
m3.add_circle_markers(df, name="Sensors",
                      layer_group=["Sensors", "status"],
                      color_col="status")
m3.configure_group("Sensors", collapsed=False)
m3

## Popups that say what you mean

`popup_fields` narrows the list, `popup_names` relabels it (matched by position).
The same pair exists for tooltips.

In [ ]:
m4 = Map()
m4.add_circle_markers(
    df, name="Sensors",
    popup_fields=["site", "reading", "status"],
    popup_names=["Site", "Reading (ppm)", "Status"],
    tooltip_fields=["site"],
)
m4

## Steering the view

The automatic fit disarms the moment anyone states a view — `center=`/`zoom=` at
construction, a `fit_bounds()` call, or panning the map — and every layer records
its own bounds, so framing something on demand needs no coordinates from you.
Scroll back up: the cell below moves the *first* map.

In [ ]:
m.fit_bounds(m.bounds_of("Sensors"), zoom_offset=-1);

## Ship it

One self-contained HTML file: layer configs, coordinate and style buffers, the
widget bundle. It opens from disk with no Python and no server — sidebar and time
playback included. (Leaflet and glify still load from unpkg when the file is
opened, so *viewing* needs internet.)

`m.to_html()` returns the same document as a string — which is also the Streamlit
story: `st.components.v1.html(m.to_html(), height=600)`.

In [ ]:
m.save("quickstart_map.html");

## Where next

- **02_data_sources** — every input format `add_*` accepts: Polars, GeoPandas,
  geostructures, GeoJSON, WKT columns, long and wide tables, raw lists.
- Styling in depth, sidebar hierarchies, layer targeting, time animation, popups,
  export, and Shiny apps each get their own notebook as the gallery grows.